In [1]:
import sqlite3
import random
import time
import requests
import pandas as pd
from datetime import date, timedelta

random.seed(7)

conn = sqlite3.connect("retail.db")
cur = conn.cursor()

cur.execute("DROP TABLE IF EXISTS orders")
cur.execute("DROP TABLE IF EXISTS customers")

cur.execute("""
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    name TEXT,
    city TEXT,
    segment TEXT
)
""")
cur.execute("""
CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    order_date TEXT,
    amount REAL,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
)
""")

cities = ["Seoul", "Busan", "Incheon", "Daegu", "Gwangju"]
segments = ["Retail", "Wholesale", "Online"]
names = ["Kim Minji","Park Jisoo","Lee Jaehyun","Choi Seoyeon","Jung Hyunwoo",
         "Kang Yuna","Yoon Doyoon","Han Soomin","Oh Jiwoo","Shin Taeyang",
         "Baek Narae","Song Junho","Moon Hana","Ahn Jiho","Jang Yerin"]

customers = [(i, name, random.choice(cities), random.choice(segments))
             for i, name in enumerate(names, start=1)]
cur.executemany("INSERT INTO customers VALUES (?,?,?,?)", customers)

start_date = date(2026, 1, 1)
order_id, orders = 1, []
for c in customers:
    for _ in range(random.randint(2, 8)):
        d = start_date + timedelta(days=random.randint(0, 270))
        amt = round(random.uniform(15000, 450000), -2)
        orders.append((order_id, c[0], d.isoformat(), amt))
        order_id += 1
cur.executemany("INSERT INTO orders VALUES (?,?,?,?)", orders)
conn.commit()

print(f"retail.db ready: {len(customers)} customers, {len(orders)} orders.")

retail.db ready: 15 customers, 73 orders.


Part A: SQL Fundamentals

In [2]:
def customers_in_city_segment(conn, city, segment):
    """
    conn: sqlite3 connection
    city: str, e.g. "Seoul"
    segment: str, e.g. "Online"
    returns: pandas DataFrame with columns name, city, segment
    """
    query = """
        SELECT name, city, segment
        FROM customers
        WHERE city = ? AND segment = ?
    """
    return pd.read_sql(query, conn, params=(city, segment))

# Test
print(customers_in_city_segment(conn, "Seoul", "Online"))

         name   city segment
0   Kang Yuna  Seoul  Online
1  Song Junho  Seoul  Online


In [3]:
def orders_with_customer_names(conn):
    """
    returns: pandas DataFrame with columns name, order_date, amount (first 10 rows)
    """
    query = """
        SELECT customers.name, orders.order_date, orders.amount
        FROM customers
        INNER JOIN orders ON customers.customer_id = orders.customer_id
        LIMIT 10
    """
    return pd.read_sql(query, conn)

print(orders_with_customer_names(conn))

         name  order_date    amount
0   Kim Minji  2026-01-26  439700.0
1   Kim Minji  2026-01-24  257100.0
2   Kim Minji  2026-03-10  141000.0
3   Kim Minji  2026-03-15  250200.0
4   Kim Minji  2026-06-07  258700.0
5  Park Jisoo  2026-04-03   59800.0
6  Park Jisoo  2026-04-07  177000.0
7  Park Jisoo  2026-02-02  260500.0
8  Park Jisoo  2026-04-16  230900.0
9  Park Jisoo  2026-08-07  353100.0


In [4]:
def high_spending_customers(conn, min_total=300000):
    """
    returns: pandas DataFrame with columns name, total_spent
    """
    query = """
        SELECT customers.name, SUM(orders.amount) AS total_spent
        FROM customers
        INNER JOIN orders ON customers.customer_id = orders.customer_id
        GROUP BY customers.customer_id, customers.name
        HAVING SUM(orders.amount) > ?
        ORDER BY total_spent DESC
    """
    return pd.read_sql(query, conn, params=(min_total,))

print(high_spending_customers(conn))

            name  total_spent
0     Jang Yerin    2167100.0
1    Yoon Doyoon    1609600.0
2     Baek Narae    1559500.0
3     Park Jisoo    1523300.0
4       Oh Jiwoo    1437300.0
5     Song Junho    1359000.0
6      Kim Minji    1346700.0
7       Ahn Jiho    1129700.0
8      Moon Hana     895700.0
9   Jung Hyunwoo     861200.0
10  Shin Taeyang     619200.0
11   Lee Jaehyun     613200.0
12    Han Soomin     427900.0
13  Choi Seoyeon     374600.0


In [5]:
def top_n_customers(conn, n=3):
    """
    returns: pandas DataFrame with columns name, total_spent, exactly n rows
    """
    query = """
        SELECT customers.name, SUM(orders.amount) AS total_spent
        FROM customers
        INNER JOIN orders ON customers.customer_id = orders.customer_id
        GROUP BY customers.customer_id, customers.name
        ORDER BY total_spent DESC
        LIMIT ?
    """
    return pd.read_sql(query, conn, params=(n,))

print(top_n_customers(conn, 3))

          name  total_spent
0   Jang Yerin    2167100.0
1  Yoon Doyoon    1609600.0
2   Baek Narae    1559500.0


Part B: Python + SQL Safely

In [6]:
def find_orders_by_customer(conn, customer_name):
    """
    conn: sqlite3 connection
    customer_name: str
    returns: list of tuples (order_id, order_date, amount)
    """
    cur = conn.cursor()
    cur.execute("""
        SELECT orders.order_id, orders.order_date, orders.amount
        FROM orders
        INNER JOIN customers ON orders.customer_id = customers.customer_id
        WHERE customers.name = ?
    """, (customer_name,))
    return cur.fetchall()

print(find_orders_by_customer(conn, "Kim Minji"))

[(1, '2026-01-26', 439700.0), (2, '2026-01-24', 257100.0), (3, '2026-03-10', 141000.0), (4, '2026-03-15', 250200.0), (5, '2026-06-07', 258700.0)]


In [7]:
def monthly_revenue(conn):
    """
    returns: pandas DataFrame with columns month, revenue
    """
    query = """
        SELECT strftime('%Y-%m', order_date) AS month,
               SUM(amount) AS revenue
        FROM orders
        GROUP BY month
        ORDER BY month
    """
    return pd.read_sql(query, conn)

print(monthly_revenue(conn))

     month    revenue
0  2026-01  1605200.0
1  2026-02  1924900.0
2  2026-03  1836600.0
3  2026-04  2277600.0
4  2026-05  1005400.0
5  2026-06  2817100.0
6  2026-07  1715100.0
7  2026-08  1906900.0
8  2026-09  1097800.0


Part C: REST APIs

In [8]:
def get_current_weather(latitude, longitude):
    """
    returns: dict (the current_weather portion of the response), or None on failure
    """
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "current_weather": True,
    }
    try:
        response = requests.get(url, params=params, timeout=10)
    except requests.exceptions.RequestException:
        return None

    if response.status_code != 200:
        return None

    data = response.json()
    return data["current_weather"]

print(get_current_weather(37.5665, 126.9780))  # Seoul

{'time': '2026-10-09T14:30', 'interval': 900, 'temperature': 16.4, 'windspeed': 2.4, 'winddirection': 297, 'is_day': 0, 'weathercode': 0}


In [9]:
def weather_for_cities(cities):
    """
    cities: list of (city_name, latitude, longitude) tuples
    returns: pandas DataFrame with a 'city' column plus the weather fields
    """
    records = []
    for city_name, lat, lon in cities:
        weather = get_current_weather(lat, lon)
        if weather is not None:
            record = dict(weather)
            record["city"] = city_name
            records.append(record)
    return pd.DataFrame(records)

test_cities = [("Seoul", 37.5665, 126.9780), ("Busan", 35.1796, 129.0756), ("Tokyo", 35.6762, 139.6503)]
print(weather_for_cities(test_cities))

               time  interval  temperature  windspeed  winddirection  is_day  \
0  2026-10-09T14:30       900         16.4        2.4            297       0   
1  2026-10-09T14:30       900         19.9        4.4             55       0   
2  2026-10-09T14:30       900         17.2        3.3            347       0   

   weathercode   city  
0            0  Seoul  
1            1  Busan  
2            0  Tokyo  


In [10]:
def weather_for_cities_paced(cities, delay_seconds=1):
    """
    Same as Problem 8, but pauses delay_seconds between each request.
    returns: pandas DataFrame with a 'city' column plus the weather fields
    """
    records = []
    for city_name, lat, lon in cities:
        weather = get_current_weather(lat, lon)
        if weather is not None:
            record = dict(weather)
            record["city"] = city_name
            records.append(record)
        time.sleep(delay_seconds)
    return pd.DataFrame(records)

print(weather_for_cities_paced(test_cities, delay_seconds=1))

               time  interval  temperature  windspeed  winddirection  is_day  \
0  2026-10-09T14:30       900         16.4        2.4            297       0   
1  2026-10-09T14:30       900         19.9        4.4             55       0   
2  2026-10-09T14:30       900         17.2        3.3            347       0   

   weathercode   city  
0            0  Seoul  
1            1  Busan  
2            0  Tokyo  


Stretch Challenge 

In [11]:
from bs4 import BeautifulSoup

def scrape_book_titles_and_prices():
    """
    returns: pandas DataFrame with columns title, price
    """
    url = "http://books.toscrape.com"
    headers = {"User-Agent": "KDU-DataScience-Student"}

    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")

    books = []
    for item in soup.select("article.product_pod"):
        books.append({
            "title": item.select_one("h3 a")["title"],
            "price": item.select_one("p.price_color").get_text(strip=True),
        })

    time.sleep(1)    # be polite
    return pd.DataFrame(books)

print(scrape_book_titles_and_prices())

                                                title    price
0                                A Light in the Attic  Â£51.77
1                                  Tipping the Velvet  Â£53.74
2                                          Soumission  Â£50.10
3                                       Sharp Objects  Â£47.82
4               Sapiens: A Brief History of Humankind  Â£54.23
5                                     The Requiem Red  Â£22.65
6   The Dirty Little Secrets of Getting Your Dream...  Â£33.34
7   The Coming Woman: A Novel Based on the Life of...  Â£17.93
8   The Boys in the Boat: Nine Americans and Their...  Â£22.60
9                                     The Black Maria  Â£52.15
10     Starving Hearts (Triangular Trade Trilogy, #1)  Â£13.99
11                              Shakespeare's Sonnets  Â£20.66
12                                        Set Me Free  Â£17.46
13  Scott Pilgrim's Precious Little Life (Scott Pi...  Â£52.29
14                          Rip it Up and Start Again  

Cleanup

In [12]:
conn.close()
print("Connection closed.")

Connection closed.
